# Data Analysis and Visualization Notebook

This notebook demonstrates common data analysis techniques and creates various types of visualizations using sample data. We'll explore data loading, cleaning, statistical analysis, and different chart types.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
plt.style.use('default')
sns.set_palette("husl")

# Configure matplotlib
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Step 1: Create Sample Dataset

We'll generate a synthetic sales dataset with multiple dimensions for analysis.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate sample sales data
n_records = 1000

# Create date range
start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)
date_range = pd.date_range(start=start_date, end=end_date, periods=n_records)

# Sample data generation
data = {
    'date': date_range,
    'product_category': np.random.choice(['Electronics', 'Clothing', 'Home & Garden', 'Books', 'Sports'], n_records),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_records),
    'sales_amount': np.random.normal(1000, 300, n_records).round(2),
    'units_sold': np.random.poisson(10, n_records),
    'customer_age': np.random.normal(35, 12, n_records).astype(int),
    'customer_satisfaction': np.random.uniform(1, 5, n_records).round(1)
}

# Create DataFrame
df = pd.DataFrame(data)

# Clean up negative sales amounts
df['sales_amount'] = np.abs(df['sales_amount'])
df['customer_age'] = np.clip(df['customer_age'], 18, 80)

print(f"Dataset created with {len(df)} records")
df.head()

## Step 2: Data Exploration and Summary Statistics

Let's examine the structure and basic statistics of our dataset.

In [ ]:
# Dataset info
print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Summary statistics
print("Summary Statistics:")
print(df.describe())

## Step 3: Time Series Analysis

Analyze sales trends over time with line plots.

In [ ]:
# Aggregate daily sales
daily_sales = df.groupby('date')['sales_amount'].sum().reset_index()

# Create time series plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Daily sales trend
ax1.plot(daily_sales['date'], daily_sales['sales_amount'], color='steelblue', alpha=0.7)
ax1.set_title('Daily Sales Trend', fontsize=14, fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('Sales Amount ($)')
ax1.grid(True, alpha=0.3)

# Monthly sales aggregation
df['month'] = df['date'].dt.to_period('M')
monthly_sales = df.groupby('month')['sales_amount'].sum()

ax2.bar(range(len(monthly_sales)), monthly_sales.values, color='lightcoral')
ax2.set_title('Monthly Sales Summary', fontsize=14, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Total Sales ($)')
ax2.set_xticks(range(len(monthly_sales)))
ax2.set_xticklabels([str(m) for m in monthly_sales.index], rotation=45)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Category Analysis

Examine sales performance across different product categories.

In [ ]:
# Sales by category
category_sales = df.groupby('product_category').agg({
    'sales_amount': ['sum', 'mean', 'count'],
    'units_sold': 'sum'
}).round(2)

category_sales.columns = ['Total Sales', 'Avg Sales', 'Transactions', 'Total Units']
category_sales = category_sales.sort_values('Total Sales', ascending=False)

print("Sales Performance by Category:")
print(category_sales)

In [ ]:
# Visualize category performance
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Pie chart for total sales
ax1.pie(category_sales['Total Sales'], labels=category_sales.index, autopct='%1.1f%%', startangle=90)
ax1.set_title('Sales Distribution by Category', fontsize=14, fontweight='bold')

# Bar chart for average sales
ax2.bar(category_sales.index, category_sales['Avg Sales'], color='skyblue')
ax2.set_title('Average Sales by Category', fontsize=14, fontweight='bold')
ax2.set_ylabel('Average Sales ($)')
ax2.tick_params(axis='x', rotation=45)

# Box plot for sales distribution
df.boxplot(column='sales_amount', by='product_category', ax=ax3)
ax3.set_title('Sales Amount Distribution by Category', fontsize=14, fontweight='bold')
ax3.set_xlabel('Product Category')
ax3.set_ylabel('Sales Amount ($)')

# Heatmap for regional performance
region_category = df.pivot_table(values='sales_amount', index='region', columns='product_category', aggfunc='mean')
sns.heatmap(region_category, annot=True, fmt='.0f', ax=ax4, cmap='YlOrRd')
ax4.set_title('Average Sales by Region and Category', fontsize=14, fontweight='bold')

plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.show()

## Step 5: Customer Demographics Analysis

Explore customer age distribution and its relationship with sales.

In [ ]:
# Customer age analysis
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

# Age distribution histogram
ax1.hist(df['customer_age'], bins=20, color='lightgreen', alpha=0.7, edgecolor='black')
ax1.set_title('Customer Age Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Age')
ax1.set_ylabel('Frequency')
ax1.grid(True, alpha=0.3)

# Age vs Sales scatter plot
ax2.scatter(df['customer_age'], df['sales_amount'], alpha=0.6, color='coral')
ax2.set_title('Age vs Sales Amount', fontsize=14, fontweight='bold')
ax2.set_xlabel('Customer Age')
ax2.set_ylabel('Sales Amount ($)')
ax2.grid(True, alpha=0.3)

# Age groups analysis
df['age_group'] = pd.cut(df['customer_age'], bins=[18, 30, 45, 60, 80], labels=['18-30', '31-45', '46-60', '60+'])
age_group_sales = df.groupby('age_group')['sales_amount'].mean()

ax3.bar(age_group_sales.index, age_group_sales.values, color='mediumpurple')
ax3.set_title('Average Sales by Age Group', fontsize=14, fontweight='bold')
ax3.set_xlabel('Age Group')
ax3.set_ylabel('Average Sales ($)')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Correlation Analysis

Examine relationships between numerical variables.

In [ ]:
# Correlation matrix
numerical_cols = ['sales_amount', 'units_sold', 'customer_age', 'customer_satisfaction']
correlation_matrix = df[numerical_cols].corr()

# Create correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.3f', cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Numerical Variables', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Correlation Matrix:")
print(correlation_matrix.round(3))

## Step 7: Advanced Visualizations

Create more sophisticated plots for deeper insights.

In [ ]:
# Create advanced visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# Violin plot for sales distribution by region
sns.violinplot(data=df, x='region', y='sales_amount', ax=ax1)
ax1.set_title('Sales Distribution by Region', fontsize=14, fontweight='bold')
ax1.set_ylabel('Sales Amount ($)')

# Scatter plot with regression line
sns.regplot(data=df, x='units_sold', y='sales_amount', ax=ax2, scatter_kws={'alpha':0.6})
ax2.set_title('Units Sold vs Sales Amount', fontsize=14, fontweight='bold')
ax2.set_xlabel('Units Sold')
ax2.set_ylabel('Sales Amount ($)')

# Stacked bar chart for category by region
region_category_sum = df.groupby(['region', 'product_category'])['sales_amount'].sum().unstack()
region_category_sum.plot(kind='bar', stacked=True, ax=ax3)
ax3.set_title('Sales by Region and Category (Stacked)', fontsize=14, fontweight='bold')
ax3.set_xlabel('Region')
ax3.set_ylabel('Total Sales ($)')
ax3.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
ax3.tick_params(axis='x', rotation=0)

# Customer satisfaction by category
sns.boxplot(data=df, x='product_category', y='customer_satisfaction', ax=ax4)
ax4.set_title('Customer Satisfaction by Category', fontsize=14, fontweight='bold')
ax4.set_xlabel('Product Category')
ax4.set_ylabel('Customer Satisfaction')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Step 8: Key Insights Summary

Summarize the main findings from our analysis.

In [ ]:
# Calculate key metrics
total_sales = df['sales_amount'].sum()
avg_transaction = df['sales_amount'].mean()
total_units = df['units_sold'].sum()
avg_satisfaction = df['customer_satisfaction'].mean()
best_category = category_sales.index[0]
best_region = df.groupby('region')['sales_amount'].sum().idxmax()
avg_customer_age = df['customer_age'].mean()

print("=" * 50)
print("KEY INSIGHTS SUMMARY")
print("=" * 50)
print(f"📊 Total Sales: ${total_sales:,.2f}")
print(f"💰 Average Transaction: ${avg_transaction:.2f}")
print(f"📦 Total Units Sold: {total_units:,}")
print(f"⭐ Average Customer Satisfaction: {avg_satisfaction:.1f}/5.0")
print(f"🏆 Best Performing Category: {best_category}")
print(f"🗺️ Best Performing Region: {best_region}")
print(f"👥 Average Customer Age: {avg_customer_age:.0f} years")
print("\n" + "=" * 50)
print("RECOMMENDATIONS:")
print("=" * 50)
print(f"• Focus marketing efforts on {best_category} products")
print(f"• Expand operations in the {best_region} region")
print(f"• Target customers aged {avg_customer_age:.0f} ± 10 years")
print("• Investigate factors driving customer satisfaction")
print("• Monitor seasonal trends for inventory planning")

## Conclusion

This notebook demonstrated comprehensive data analysis techniques including:

- **Data Generation**: Created realistic synthetic sales data
- **Exploratory Analysis**: Examined data structure and summary statistics
- **Time Series Visualization**: Analyzed trends over time
- **Category Analysis**: Compared performance across product categories
- **Demographic Insights**: Explored customer age patterns
- **Correlation Analysis**: Identified relationships between variables
- **Advanced Visualizations**: Used various chart types for deeper insights

### Next Steps

1. **Predictive Modeling**: Build models to forecast future sales
2. **Customer Segmentation**: Use clustering techniques to identify customer groups
3. **A/B Testing**: Design experiments to test marketing strategies
4. **Real-time Dashboard**: Create interactive visualizations for ongoing monitoring
5. **Data Pipeline**: Automate data collection and analysis processes

This analysis framework can be adapted for various business scenarios and real datasets.